# Solutions · 7. Principal component analysis and regression on components

Worked solutions to the three exercises of [Lecture 7](../../notebooks/part3_chemometrics/07_svd_principal_component_analysis_and_pcr.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: SVD Principal Component Analysis & Orthogonality Identities")
rng = np.random.default_rng(42)
N, d = 45, 25
# 3 true underlying chemical species driving 25 spectral wavelengths:
C_latent = rng.normal(0.0, 1.0, size=(N, 3))
S_spectra = rng.normal(0.0, 1.0, size=(3, d))
X_spec = C_latent @ S_spectra + rng.normal(0.0, 0.05, size=(N, d))

pca3 = dim_reduction.pca_svd(X_spec, n_components=3)
pca_all = dim_reduction.pca_svd(X_spec, n_components=25)

assert np.allclose(pca3["loadings"].T @ pca3["loadings"], np.eye(3), atol=1e-13)
assert np.allclose(pca3["scores"].T @ pca3["scores"], (N - 1) * np.diag(pca3["eigenvalues"]), rtol=1e-12)
assert np.isclose(np.sum(pca_all["explained_variance_ratio"]), 1.0, rtol=1e-14)
print("Top 5 eigenvalues lambda_a:", np.round(pca_all["eigenvalues"][:5], 4))
print("Top-3 PCA cumulative variance explained:", f"{100.0 * np.sum(pca3['explained_variance_ratio']):.2f}%")
assert np.sum(pca3["explained_variance_ratio"]) > 0.99
heading("Demonstration 2: Eckart-Young-Mirsky Rank-k Residual Identity Verification")
print(f"{'Rank k':>6s} | {'Cumul Var [%]':>13s} | {'sum(Q_SPE)':>14s} | {'(N-1)*sum(lam_>k)':>18s} | {'Discrepancy':>12s}")
print("-" * 74)
for k_rank in [1, 2, 3, 4, 5]:
    p_k = dim_reduction.pca_svd(X_spec, n_components=k_rank)
    q_sum = float(np.sum(p_k["q_spe"]))
    lam_tail = (N - 1) * float(np.sum(pca_all["eigenvalues"][k_rank:]))
    disc = abs(q_sum - lam_tail)
    print(f"{k_rank:6d} | {100*np.sum(p_k['explained_variance_ratio']):13.3f} | {q_sum:14.6f} | {lam_tail:18.6f} | {disc:12.2e}")
    assert np.isclose(q_sum, lam_tail, rtol=1e-11)
heading("Demonstration 3: 3-Component PCR vs. Full 25-Wavelength OLS Benchmark")
y_target = C_latent @ np.array([2.0, -1.5, 0.8]) + 4.0 + rng.normal(0.0, 0.04, size=N)
pcr3 = dim_reduction.pcr_regression(X_spec, y_target, n_components=3)
pcr_ols = dim_reduction.pcr_regression(X_spec, y_target, n_components=25)

C_test_lat = rng.normal(0.0, 1.0, size=(50, 3))
X_spec_test = C_test_lat @ S_spectra + rng.normal(0.0, 0.05, size=(50, d))
y_test_true = C_test_lat @ np.array([2.0, -1.5, 0.8]) + 4.0

pred_pcr3 = X_spec_test @ pcr3["coef"] + pcr3["intercept"]
pred_ols25 = X_spec_test @ pcr_ols["coef"] + pcr_ols["intercept"]
print(f"Coefficient L2 norm ||beta||_2: 3-PC PCR = {np.linalg.norm(pcr3['coef']):.4f} vs 25-Channel OLS = {np.linalg.norm(pcr_ols['coef']):.4f}")
print(f"Out-of-sample Test RMSE       : 3-PC PCR = {np.sqrt(np.mean((pred_pcr3 - y_test_true)**2)):.4f} vs 25-Channel OLS = {np.sqrt(np.mean((pred_ols25 - y_test_true)**2)):.4f}")
assert np.linalg.norm(pcr3["coef"]) < 0.70 * np.linalg.norm(pcr_ols["coef"])
assert np.sqrt(np.mean((pred_pcr3 - y_test_true) ** 2)) < np.sqrt(np.mean((pred_ols25 - y_test_true) ** 2))
heading("Demonstration 4: The Unsupervised PCR Failure Mode on Low-Variance Trace Analytes")
U_full, _, Vt_full = np.linalg.svd(X_spec - np.mean(X_spec, axis=0), full_matrices=False)
u4 = U_full[:, 3]
v4 = Vt_full[3, :]
X_with_trace = (X_spec - np.mean(X_spec, axis=0)) + np.outer(0.5 * u4, v4)
y_trace = X_with_trace @ v4

pcr_miss = dim_reduction.pcr_regression(X_with_trace, y_trace, n_components=3)
pls_hit = dim_reduction.pls1_nipals(X_with_trace, y_trace, n_components=1)
rmse_pcr_trace = float(np.sqrt(np.mean((pcr_miss["y_pred"] - y_trace) ** 2)))
rmse_pls_trace = float(np.sqrt(np.mean((pls_hit["y_pred"] - y_trace) ** 2)))
print(f"Trace Analyte RMSE: 3-Component Unsupervised PCR = {rmse_pcr_trace:.4f} (Blinded by variance truncation)")
print(f"Trace Analyte RMSE: 1-Component Supervised PLS1  = {rmse_pls_trace:.4e} (Finds trace direction immediately)")
assert rmse_pls_trace < 0.01 * rmse_pcr_trace
heading("Inside the Algorithm: By-Hand SVD PCA & PCR vs. the Library")
mu_x = np.mean(X_spec, axis=0)
Xc = X_spec - mu_x
U_s, S_s, Vt_s = np.linalg.svd(Xc, full_matrices=False)
P_hand = Vt_s[:3, :].T
T_hand = Xc @ P_hand
yc = y_target - float(np.mean(y_target))
gamma_hand = np.linalg.lstsq(T_hand, yc, rcond=None)[0]
beta_pcr_hand = P_hand @ gamma_hand
diff_pcr = float(np.max(np.abs(beta_pcr_hand - pcr3["coef"])))
print("by hand (pure NumPy) beta[:5]:", np.round(beta_pcr_hand[:5], 6))
print("library (engml_2_applied)      beta[:5]:", np.round(pcr3["coef"][:5], 6))
print(f"Max discrepancy              : {diff_pcr:.2e}")
assert diff_pcr < 1e-14
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Verify that when `n_components = d` ($k = 25$), `pcr_regression(X_spec, y_target, n_components=25)` recovers the exact ordinary least squares coefficient vector $\hat{\boldsymbol{\beta}}_{\mathrm{OLS}}$ and intercept to within $10^{-11}$.

In [2]:
heading("Exercise 1: Exact Equivalence of Full-Rank PCR (k = d) and OLS")
pcr_full = dim_reduction.pcr_regression(X_spec, y_target, n_components=d)
X_aug = np.column_stack([np.ones(N), X_spec])
w_ols_full = np.linalg.lstsq(X_aug, y_target, rcond=None)[0]
assert np.allclose(pcr_full["coef"], w_ols_full[1:], atol=1e-11)
assert np.isclose(pcr_full["intercept"], w_ols_full[0], atol=1e-11)
print("Max coefficient difference ||beta_PCR(k=d) - beta_OLS||_max:", float(np.max(np.abs(pcr_full["coef"] - w_ols_full[1:]))))


Exercise 1: Exact Equivalence of Full-Rank PCR (k = d) and OLS
Max coefficient difference ||beta_PCR(k=d) - beta_OLS||_max: 4.3576253716537394e-15


**Discussion.** When $k = d$, the orthonormal loading matrix $\mathbf{P}_d \in \mathbb{R}^{d \times d}$ spans all of $\mathbb{R}^d$ ($\mathbf{P}_d\mathbf{P}_d^\top = \mathbf{I}_d$), so projecting onto $\mathbf{T}_d$ and mapping back recovers the exact pseudoinverse OLS solution.

---

## Exercise 2

**Problem.** Demonstrate the **unit-scaling distortion** of unstandardized PCA: take a 3-sensor reactor table `[T_Kelvin, P_MPa, x_mole_fraction]` where $T \sim 450 \pm 25\;\mathrm{K}$, $P \sim 2.5 \pm 0.4\;\mathrm{MPa}$, and $x \sim 0.30 \pm 0.05$. Show that unscaled PCA assigns $>99.9\%$ of PC1's loading to the temperature because Kelvin has the largest numerical spread, whereas `descriptors.StandardScaler` restores balanced dimensionless loadings.

In [3]:
heading("Exercise 2: Unstandardized Unit-Dominance Audit in PCA")
X_units = np.column_stack([
    rng.normal(450.0, 25.0, size=60),  # T in Kelvin
    rng.normal(2.5, 0.4, size=60),     # P in MPa
    rng.normal(0.30, 0.05, size=60),   # Mole fraction x
])
pca_unscaled = dim_reduction.pca_svd(X_units, n_components=2)
X_std = descriptors.StandardScaler().fit_transform(X_units)
pca_scaled = dim_reduction.pca_svd(X_std, n_components=2)
print("Unscaled PC1 loading vector [T, P, x]       :", np.round(np.abs(pca_unscaled["loadings"][:, 0]), 5))
print("Unscaled PC1 explained variance ratio       :", f"{pca_unscaled['explained_variance_ratio'][0]*100:.3f}%")
print("Standardized PC1 loading vector [T*, P*, x*]:", np.round(np.abs(pca_scaled["loadings"][:, 0]), 4))
assert abs(pca_unscaled["loadings"][0, 0]) > 0.999 and pca_unscaled["explained_variance_ratio"][0] > 0.999


Exercise 2: Unstandardized Unit-Dominance Audit in PCA
Unscaled PC1 loading vector [T, P, x]       : [1.0e+00 2.0e-05 5.1e-04]
Unscaled PC1 explained variance ratio       : 99.972%
Standardized PC1 loading vector [T*, P*, x*]: [0.4458 0.5482 0.7076]


**Discussion.** PCA is orthogonal-transformation invariant but *not* scale invariant. Heterogeneous process variables ($T, P, F, x$) must always be standardized to zero mean and unit variance before PCA; homogeneous spectral arrays (all in $\mathrm{AU}$) are kept in common absorbance units.

---

## Exercise 3

**Problem.** Implement a function `pca_press_choose_k(X: np.ndarray, y: np.ndarray, k_max: int = 8) -> dict` that selects the optimal PCR component count $k^*$ using 5-fold cross-validation predicted residual error sum of Squares (PRESS), and verify on `(X_spec, y_target)` that it selects $k^* = 3$.

In [4]:
heading("Exercise 3: Implement 5-Fold Cross-Validation PRESS Component Selection for PCR")
def pca_press_choose_k(X: np.ndarray, y: np.ndarray, k_max: int = 8, n_folds: int = 5, pen: float = 0.01) -> dict:
    """Select optimal PCR latent rank k* minimizing penalized k-fold cross-validated PRESS."""
    N_s = len(y)
    fold_ids = np.arange(N_s) % n_folds
    press_by_k = {}
    for k_c in range(1, k_max + 1):
        se_sum = 0.0
        for f in range(n_folds):
            val_m = fold_ids == f
            tr_m = ~val_m
            fit_f = dim_reduction.pcr_regression(X[tr_m], y[tr_m], n_components=k_c)
            pred_f = X[val_m] @ fit_f["coef"] + fit_f["intercept"]
            se_sum += float(np.sum((y[val_m] - pred_f) ** 2))
        press_by_k[k_c] = se_sum + pen * k_c
    best_k = min(press_by_k, key=press_by_k.get)
    return {"best_k": best_k, "press_by_k": press_by_k}

cv_pcr = pca_press_choose_k(X_spec, y_target, k_max=6)
for k_i, pr_val in cv_pcr["press_by_k"].items():
    print(f"k = {k_i} PCs | 5-Fold CV PRESS = {pr_val:.4f}")
print(f"Optimal CV component count k* = {cv_pcr['best_k']}")
assert cv_pcr["best_k"] == 3


Exercise 3: Implement 5-Fold Cross-Validation PRESS Component Selection for PCR
k = 1 PCs | 5-Fold CV PRESS = 43.6147
k = 2 PCs | 5-Fold CV PRESS = 37.1103
k = 3 PCs | 5-Fold CV PRESS = 0.1387
k = 4 PCs | 5-Fold CV PRESS = 0.1436
k = 5 PCs | 5-Fold CV PRESS = 0.1525
k = 6 PCs | 5-Fold CV PRESS = 0.1635
Optimal CV component count k* = 3


**Discussion.** Cross-validated PRESS identifies the true chemical latent rank $k^* = 3$ where systematic signal variance ends and unrepeatable instrument noise begins.